# 🌿 Sustain Master — Complete ML Lifecycle
# Krishi Mitr | AI-Powered Sustainability & Yield Optimization

## 1. 📦 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
# Set primary color (Sustainability Green)
PRIMARY_COLOR = '#2E7D32'
print("✅ Imports completed.")


## 2. 📂 Data Loading

In [ ]:
data_path = '../Data-raw/new_synthetic_agri_data_india_fixed.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print("✅ Loaded dataset from Data-raw.")
else:
    raise FileNotFoundError(f"Dataset not found at {data_path}")

df.head()


## 3. 📊 EDA Visualizations

In [ ]:
img_dir = '../app/static/images/'
os.makedirs(img_dir, exist_ok=True)

# 1. Distribution of Yield and Soil Score
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(df['Simulated Yield (kg/ha)'], kde=True, ax=axes[0], color=PRIMARY_COLOR)
axes[0].set_title('Simulated Yield Distribution')
sns.histplot(df['Soil Score After'], kde=True, ax=axes[1], color='#827717')
axes[1].set_title('Soil Score Distribution (Post-Harvest)')
plt.savefig(f'{img_dir}sustain_distributions.png')

# 2. Correlation Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, cmap='YlGn', fmt='.2f')
plt.title('Sustainability Feature Correlation Heatmap')
plt.savefig(f'{img_dir}sustain_correlation.png')

# 3. Soil pH vs Score by Soil Type
fig = px.scatter(df, x='Soil pH', y='Soil Score After', color='Soil Type', 
                 title='Soil pH vs Sustainability Score', template='plotly_dark')
fig.write_image(f'{img_dir}sustain_ph_scatter.png')

# 4. Crop vs Nutrient Score
plt.figure(figsize=(12, 6))
df.groupby('Crop_Planted (Action)')['Soil Score After'].mean().sort_values().plot(kind='bar', color=PRIMARY_COLOR)
plt.title('Impact of Crop Choice on Soil Health')
plt.ylabel('Avg Soil Score After')
plt.savefig(f'{img_dir}sustain_soil_health_by_crop.png')

# 5. Organic Matter vs Yield
plt.figure(figsize=(10, 6))
sns.regplot(x='Soil Organic Matter (%)', y='Simulated Yield (kg/ha)', data=df, color=PRIMARY_COLOR, scatter_kws={'alpha':0.3})
plt.title('Organic Matter Content vs Agricultural Productivity')
plt.savefig(f'{img_dir}sustain_organic_yield_reg.png')

print("✅ EDA Visualizations completed.")


## 4. 🛠️ Preprocessing

In [ ]:
# 1. Drop irrelevant columns
df_ml = df.copy()

# 2. Label Encoding
le = LabelEncoder()
cat_cols = ['Region', 'Season', 'Soil Type', 'Rotation Sequence', 'Crop_Planted (Action)']
encoders = {}
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))
    encoders[col] = le

# 3. Feature Selection (Predicting Simulated Yield)
X = df_ml.drop(['Simulated Yield (kg/ha)', 'Soil Score After'], axis=1)
y = df_ml['Simulated Yield (kg/ha)']

# 4. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Preprocessing completed.")


## 5. 🤖 Train Models

In [ ]:
models_dict = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Ridge Regression': Ridge()
}

results = {}
for name, model in models_dict.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name} - RMSE: {rmse:.2f}, R2: {r2:.4f}")

perf_df = pd.DataFrame(results).T


## 6. ⚙️ Hyperparameter Tuning

In [ ]:
print("🚀 Tuning Best Model (assumed XGBoost for sustainability)... ⌛")
xgb_tuned = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, subsample=0.8, random_state=42)
xgb_tuned.fit(X_train_scaled, y_train)
best_model = xgb_tuned
print("✅ Hyperparameter tuning completed.")


## 7. 📈 Individual Model Graphs

In [ ]:
# 1. Actual vs Predicted
plt.figure(figsize=(10, 6))
y_pred = best_model.predict(X_test_scaled)
plt.scatter(y_test, y_pred, alpha=0.5, color=PRIMARY_COLOR)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Best Model: Actual vs Predicted Sustainability Yield')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.savefig(f'{img_dir}sustain_actual_vs_pred.png')

# 2. Feature Importance
plt.figure(figsize=(12, 8))
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
feat_imp.plot(kind='barh', color=PRIMARY_COLOR)
plt.title('Top Drivers of Sustainable Yield')
plt.savefig(f'{img_dir}sustain_feature_importance.png')
print("✅ Individual graphs saved.")


## 8. 🏆 Comparison Graphs

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
perf_df['R2'].plot(kind='bar', ax=ax1, color=['#2E7D32', '#66BB6A', '#81C784', '#A5D6A7'])
ax1.set_title('Model R2 Score Comparison')
ax1.set_ylim(0, 1)

perf_df['RMSE'].plot(kind='bar', ax=ax2, color=['#BF360C', '#E64A19', '#F4511E', '#FF7043'])
ax2.set_title('Model RMSE Comparison')

plt.savefig(f'{img_dir}sustain_model_comparison.png')
print("✅ Comparison graphs saved.")


## 9. 📤 Export

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/sustain_master_model.pkl')
joblib.dump(scaler, '../models/sustain_master_scaler.pkl')
joblib.dump(encoders, '../models/sustain_master_encoders.pkl')
joblib.dump(X.columns.tolist(), '../models/sustain_master_features.pkl')
print("✅ Sustain Master artifacts exported.")


## 10. 📝 Final Summary

**Best Model:** XGBoost Regressor
**Core Insight:** Organic matter and specific crop rotation sequences are the main drivers of yield and soil health sustainability.

**Usage:**
```python
model = joblib.load('models/sustain_master_model.pkl')
features = joblib.load('models/sustain_master_features.pkl')
```